In [1]:
"""Data validation and preparation for twfeiw estimators."""

from __future__ import annotations

from dataclasses import dataclass

import pandas as pd
from pandas.api.types import is_integer_dtype, is_numeric_dtype


UNIT_CODE_COL = "_twfeiw_unit_code"
FIRST_TREAT_COL = "_twfeiw_first_treat_time"
EVENT_TIME_COL = "_twfeiw_event_time"
EVER_TREATED_COL = "_twfeiw_ever_treated"


@dataclass(frozen=True)
class PreparedPanel:
    """Validated panel data plus internal columns needed by estimators."""

    data: pd.DataFrame
    unit: str
    time: str
    outcome: str
    treatment: str
    unit_code_col: str
    first_treat_col: str
    event_time_col: str
    ever_treated_col: str


def prepare_panel(
    data: pd.DataFrame,
    *,
    unit: str = "unit_id",
    time: str = "time",
    outcome: str = "y",
    treatment: str = "treatment",
) -> PreparedPanel:
    """Validate raw panel data and add treatment-timing variables.

    The first version expects a staggered-adoption panel with integer-like time
    periods, one row per unit-time pair, numeric outcomes, and binary absorbing
    treatment.
    """

    if not isinstance(data, pd.DataFrame):
        raise TypeError("data must be a pandas DataFrame")

    required = [unit, time, outcome, treatment]
    _check_required_columns(data, required)
    _check_internal_columns_available(data)
    _check_no_missing_required(data, required)
    _check_numeric_outcome(data, outcome)
    _check_integer_time(data, time)
    _check_unique_unit_time(data, unit, time)
    _check_binary_treatment(data, treatment)

    prepared = data.copy()
    prepared[treatment] = prepared[treatment].astype(int)
    prepared = _add_unit_codes(prepared, unit)
    prepared = prepared.sort_values([UNIT_CODE_COL, time]).reset_index(drop=True)

    _check_absorbing_treatment(prepared, treatment)
    prepared = _add_treatment_timing(prepared, time, treatment)

    return PreparedPanel(
        data=prepared,
        unit=unit,
        time=time,
        outcome=outcome,
        treatment=treatment,
        unit_code_col=UNIT_CODE_COL,
        first_treat_col=FIRST_TREAT_COL,
        event_time_col=EVENT_TIME_COL,
        ever_treated_col=EVER_TREATED_COL,
    )

# -> None means the function is expected not to return anything

# Checks if all required cols are present
def _check_required_columns(data: pd.DataFrame, required: list[str]) -> None:
    missing = [column for column in required if column not in data.columns]
    if missing:
        raise ValueError(f"data is missing required columns: {missing}")

# This is to make sure that the initial df doesn't already have the extra cols it is supposed to make later
def _check_internal_columns_available(data: pd.DataFrame) -> None:
    internal_columns = [
        UNIT_CODE_COL,
        FIRST_TREAT_COL,
        EVENT_TIME_COL,
        EVER_TREATED_COL,
    ]
    collisions = [column for column in internal_columns if column in data.columns]
    if collisions:
        raise ValueError(f"data contains reserved internal columns: {collisions}")

# data shouldn't contain any missing values
def _check_no_missing_required(data: pd.DataFrame, required: list[str]) -> None:
    if data[required].isna().any().any():
        raise ValueError("required columns cannot contain missing values")

# outcome variable numeric
def _check_numeric_outcome(data: pd.DataFrame, outcome: str) -> None:
    if not is_numeric_dtype(data[outcome]):
        raise ValueError("outcome column must be numeric")

# make sure time is integer
def _check_integer_time(data: pd.DataFrame, time: str) -> None:
    if not is_integer_dtype(data[time]):
        raise ValueError("time column must contain integer-like periods")

# there shouldn't be a duplicated unit - time pair
def _check_unique_unit_time(data: pd.DataFrame, unit: str, time: str) -> None:
    if data.duplicated([unit, time]).any():
        raise ValueError("data must contain at most one row per unit-time pair")

# treatment must be in {0,1}
def _check_binary_treatment(data: pd.DataFrame, treatment: str) -> None:
    values = set(data[treatment].dropna().unique())
    if not values.issubset({0, 1}):
        raise ValueError("treatment must be binary with values 0 and 1")

# make unit int codes - transflorms any int or str of unit_id to numeric codes
def _add_unit_codes(data: pd.DataFrame, unit: str) -> pd.DataFrame:
    prepared = data.copy()
    prepared[UNIT_CODE_COL] = pd.factorize(prepared[unit])[0]
    return prepared

# makes sure if treatment is starting from 1, it also continues to be 1 for the unit
def _check_absorbing_treatment(data: pd.DataFrame, treatment: str) -> None:
    diff = data.groupby(UNIT_CODE_COL, sort=False)[treatment].diff()
    if (diff == -1).any():
        raise ValueError("treatment must be absorbing within each unit")


# adds the columns of event time and boolean column of ever_treated
def _add_treatment_timing(
    data: pd.DataFrame,
    time: str,
    treatment: str,
) -> pd.DataFrame:
    prepared = data.copy()

    prepared[EVER_TREATED_COL] = (
        prepared.groupby(UNIT_CODE_COL, sort=False)[treatment]
        .transform("max")
        .astype(bool)
    )

    treated_rows = prepared[prepared[treatment] == 1]
    first_treat = treated_rows.groupby(UNIT_CODE_COL, sort=False)[time].min()

    prepared[FIRST_TREAT_COL] = (
        prepared[UNIT_CODE_COL].map(first_treat).astype("Int64")
    )
    prepared[EVENT_TIME_COL] = (
        prepared[time] - prepared[FIRST_TREAT_COL]
    ).astype("Int64")

    return prepared


# for future - add check on unit_id

In [2]:

# a valid df
def valid_data() -> pd.DataFrame:
    return pd.DataFrame(
        {
            "unit_id": ["A", "A", "A", "B", "B", "B", "C", "C", "C", "D", "D", "D", "D"],
            "time": [2000, 2001, 2002, 2000, 2001, 2002, 2000, 2001, 2002, 2000, 2001, 2002, 2003],
            "y": [1.0, 1.5, 2.0, 3.0, 3.5, 4.0, 2.0, 2.5, 3.0, 3.0, 3.0, 3.5, 3.5],
            "treatment": [0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
        }
    )


In [4]:
df = valid_data()
df

,unit_id,time,y,treatment
0,A,2000,1.0,0
1,A,2001,1.5,1
2,A,2002,2.0,1
3,B,2000,3.0,0
4,B,2001,3.5,0
5,B,2002,4.0,1
6,C,2000,2.0,0
7,C,2001,2.5,0
8,C,2002,3.0,0
9,D,2000,3.0,0


In [48]:
prepared_panel = prepare_panel(df)
data = prepared_panel.data
panel = prepared_panel

In [47]:
"""Regression design construction for twfeiw estimators."""

from __future__ import annotations

from dataclasses import dataclass

import pandas as pd



@dataclass(frozen=True)
class DesignMatrix:
    """Named regression inputs plus metadata for later estimation results."""

    outcome: pd.Series
    regressors: pd.DataFrame
    model: str
    outcome_variable: str
    effect_cols: list[str]
    fixed_effect_cols: list[str]
    regressor_cols: list[str]
    reference_unit: object
    reference_time: int


def build_twfe_design(panel: PreparedPanel) -> DesignMatrix:
    """Build an explicit dummy-variable design for the standard TWFE model."""

    if not isinstance(panel, PreparedPanel):
        raise TypeError("panel must be a PreparedPanel")

    data = panel.data.reset_index(drop=True)

    outcome = _build_outcome(data, panel.outcome)
    intercept = _build_intercept(len(data))
    treatment = _build_treatment(data, panel.treatment)

    unit_dummies, reference_unit = _build_unit_fe_dummies(data, panel)
    time_dummies, reference_time = _build_time_fe_dummies(data, panel.time)

    regressors = pd.concat(
        [intercept, treatment, unit_dummies, time_dummies],
        axis=1,
    )

    fixed_effect_cols = list(unit_dummies.columns) + list(time_dummies.columns)

    return DesignMatrix(
        outcome=outcome,
        regressors=regressors,
        model="twfe",
        outcome_variable=panel.outcome,
        effect_cols=[panel.treatment],
        fixed_effect_cols=fixed_effect_cols,
        regressor_cols=list(regressors.columns),
        reference_unit=reference_unit,
        reference_time=reference_time,
    )


def _build_outcome(data: pd.DataFrame, outcome: str) -> pd.Series:
    return data[outcome].astype(float).rename(outcome)


def _build_intercept(nobs: int) -> pd.DataFrame:
    return pd.DataFrame({"const": [1.0] * nobs})


def _build_treatment(data: pd.DataFrame, treatment: str) -> pd.DataFrame:
    return data[[treatment]].astype(float)


def _build_unit_fe_dummies(
    data: pd.DataFrame,
    panel: PreparedPanel,
) -> tuple[pd.DataFrame, object]:
    unit_codes = sorted(data[panel.unit_code_col].unique())
    reference_unit_code = unit_codes[0]
    reference_unit = data.loc[
        data[panel.unit_code_col] == reference_unit_code,
        panel.unit,
    ].iloc[0]

    dummies = _build_dummies(
        data[panel.unit_code_col],
        categories=unit_codes,
        prefix="unit_fe",
    )
    return dummies, reference_unit


def _build_time_fe_dummies(
    data: pd.DataFrame,
    time: str,
) -> tuple[pd.DataFrame, int]:
    time_periods = sorted(data[time].unique())
    reference_time = int(time_periods[0])

    dummies = _build_dummies(
        data[time],
        categories=time_periods,
        prefix="time_fe",
    )
    return dummies, reference_time


def _build_dummies(
    values: pd.Series,
    *,
    categories: list[object],
    prefix: str,
) -> pd.DataFrame:
    categorical = pd.Categorical(values, categories=categories, ordered=True)
    return pd.get_dummies(
        categorical,
        prefix=prefix,
        drop_first=True,
        dtype=float,
    )

In [49]:
unit_codes = sorted(data[panel.unit_code_col].unique())
values = data[panel.unit_code_col]
categories = unit_codes
prefix = "unit_fe"


In [51]:
unit_codes

[0, 1, 2, 3]

In [60]:
categorical = pd.Categorical(values, categories=categories, ordered=True)
categorical

[0, 0, 0, 1, 1, ..., 2, 3, 3, 3, 3]
Length: 13
Categories (4, int64): [0 < 1 < 2 < 3]

In [63]:
pd.get_dummies(
        categorical,
        prefix=prefix,
        drop_first=True,
        dtype=float,
    )

,unit_fe_1,unit_fe_2,unit_fe_3
0,0.0,0.0,0.0
1,0.0,0.0,0.0
2,0.0,0.0,0.0
3,1.0,0.0,0.0
4,1.0,0.0,0.0
5,1.0,0.0,0.0
6,0.0,1.0,0.0
7,0.0,1.0,0.0
8,0.0,1.0,0.0
9,0.0,0.0,1.0


In [68]:
design = build_twfe_design(panel)

In [70]:
design.regressors["treatment"].tolist()

[0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]